In [ ]:
import token
from turtle import left
from sympy import per
from tokenizers import InputSequence
import torch
import torch.nn.functional as F
import numpy as np

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

from matplotlib import pyplot as plt
import os
from transformers import AutoTokenizer
import math

llama_model = "meta-llama/Llama-2-7b-hf"
model = "llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(
    llama_model,
    use_fast=False,        
)
dataset_name="wikitext"
start = 0
kv = torch.load(f"../{model}_{dataset_name}_st{start}.pt", weights_only=False)
model_config = kv["model_config"]
kv_info = kv["before_rope"]
rope_qkv = kv["after_rope"]
Wo = kv["Wo"] # Shape (hidden_size, hidden_size)
Wlm = kv["Wlm"] # Shape (vocab_size, hidden_size)
inputs = kv["input"]
print("model_config", model_config)

/scratch1/liankewei/miniconda3/envs/nanogpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model_config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.1.0",
  "use_cache": true,
  "vocab_size": 32000
}



In [ ]:
L = model_config.num_hidden_layers

def get_attention_map_after_rope(layer_idx, head_idx=0, causal=True, dtype=torch.float32):
    """
    返回: attn [seq_len, seq_len] (softmax 后)
    """
    Q = rope_qkv[layer_idx]['q']  # [B, nh, seq, hd]
    K = rope_qkv[layer_idx]['k']  # [B, nh, seq, hd]

    q = Q[0, head_idx].to(dtype)  # [seq, hd]
    k = K[0, head_idx].to(dtype)  # [seq, hd]

    hd = q.shape[-1]
    scores = (q @ k.T) / math.sqrt(hd)  # [seq, seq]

    if causal:
        seq = scores.shape[0]
        mask = torch.triu(torch.ones(seq, seq, device=scores.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))

    attn = torch.softmax(scores, dim=-1)
    return attn
attention_score = {}
per_token_attention = torch.zeros(inputs['input_ids'].shape[1])  # [seq_len]
avg_t0i = 0
avg_tii = 0
avg_t10 = 0
for head_idx in range(model_config.num_attention_heads):
    attention_score[head_idx] = get_attention_map_after_rope(layer_idx=L-1, head_idx=head_idx, causal=True)
    per_token_attention = per_token_attention + attention_score[head_idx].sum(dim=0)  # 每个 token 的总 attention 权重
    avg_t0i += attention_score[head_idx][:, 0].mean().item() # 每个 token 对第一个 token 的 attention 权重的平均值
    avg_tii += attention_score[head_idx].diagonal().mean().item() # 每个 token 对自己的 attention 权重的平均值
    # [i-10, i] 的 attention 权重的平均值
    avg_t10 += torch.stack([attention_score[head_idx][i, i-10:i].sum() for i in range(10, attention_score[head_idx].shape[1])]).mean().item()
avg_t0i /= model_config.num_attention_heads
avg_tii /= model_config.num_attention_heads
avg_t10 /= model_config.num_attention_heads

print("per_token_attention", per_token_attention[:20])
print("avg_t0i", avg_t0i)
print("avg_tii", avg_tii)
print("avg_t10", avg_t10)

per_token_attention tensor([4.2970e+04, 6.3508e+01, 7.3902e+01, 9.8864e+01, 3.4580e+01, 6.4898e+01,
        5.4447e+01, 7.1360e+01, 7.6534e+01, 2.4084e+02, 8.1327e+01, 8.1376e+01,
        3.5125e+01, 2.5143e+01, 4.3266e+01, 5.0483e+01, 3.8251e+01, 3.8630e+01,
        3.5168e+01, 7.3621e+01])
avg_t0i 0.3278329891181784
avg_tii 0.054476236866321415
avg_t10 0.09855807345593348


In [ ]:

V:torch.Tensor = rope_qkv[L-1]["v"]  # last layer's value, shape (B,nh,seq_len,hd)
V = V[0] # (nh,seq_len,hd)
V_attn =  {}
for head_idx in range(model_config.num_attention_heads):
    V_attn[head_idx] = attention_score[head_idx].unsqueeze(-1) * V[head_idx].unsqueeze(0)  # (B,seq_len,hd)

B, seq_len, nh, hd = V.shape
print("V shape:", V.shape)
V = V.view(B, seq_len, nh * hd)  # (B,seq_len,nh*hd)
print("Wo shape:", Wo.shape)
Z_hidden = V @ Wo.T  # (B,seq_len,hidden_size)
"""
There should be non-linear MLP here, ignoring for simplicity
Z_hidden = F.relu(Z_hidden @ W1.T) @ W2.T  # (B,seq_len,hidden_size)
"""
Z_final = Z_hidden @ Wlm.T  # (B,seq_len, vocab_size)
Z_hidden_np = Z_hidden[0].cpu().numpy()  # (seq_len, hidden_size)
Z_hidden_np = Z_hidden_np.astype(np.float32)
print("Output shape:", Z_final.shape)




V shape: torch.Size([1, 4096, 32, 128])
Wo shape: torch.Size([4096, 4096])
Output shape: torch.Size([1, 4096, 32000])


We found that supporting vectors are not the ones selected by attention.

In [8]:
print(inputs['input_ids'].shape)
print(inputs['input_ids'][0][:10])
print(tokenizer.decode(inputs['input_ids'][0][:10]))

torch.Size([1, 4096])
tensor([    1, 29871,   353,  4755,   350,  5059,   357,   353, 29871,    13],
       device='cuda:0')
<s>  = Robert Boulter = 



In [25]:


"""
finding the optimal alpha star for a given token
"""
def solve_alpha_star_logits(Z_logit: torch.Tensor, y_id: int, steps=300, lr=0.05):
    """
    Z_logit: (n, V) float32
    y_id: int
    returns alpha_star (n,)
    """
    device = Z_logit.device
    Z_logit = Z_logit.to(torch.float32)
    n, V = Z_logit.shape
    w = torch.zeros(n, device=device, requires_grad=True)
    opt = torch.optim.Adam([w], lr=lr)

    for _ in range(steps):
        alpha = F.softmax(w, dim=0)              # (n,)
        s = alpha @ Z_logit                      # (V,)
        loss = -s[y_id] + torch.logsumexp(s, dim=0)

        opt.zero_grad()
        loss.backward()
        opt.step()

    return F.softmax(w, dim=0).detach()

y_pos = 3500

alpha = solve_alpha_star_logits(Z_final[0][:y_pos], inputs['input_ids'][0][y_pos].item())
print(alpha)

tensor([1.7532e-01, 1.6518e-06, 1.0035e-07,  ..., 8.0036e-08, 1.7005e-07,
        1.1337e-07])


In [26]:
print(alpha.max(), alpha.argmax(), alpha.shape)
s = alpha @ Z_final[0][:y_pos].to(torch.float32)  # (V,)
print(s.shape)
s = F.softmax(s,dim=0)
token = inputs['input_ids'][0][y_pos]
print(s.max(), s.argmax(),s[token],token)

tensor(0.4965) tensor(276) torch.Size([3500])
torch.Size([32000])
tensor(0.0134) tensor(29871) tensor(0.0009) tensor(540, device='cuda:0')
